In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\LENOVO\\Desktop\\end-to-end ML\\Wine_Quality_Prediction'

In [13]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    n_estimators: int  # ✅ Add this
    max_depth: int  # ✅ Add this
    min_samples_split: int  # ✅ Add this
    min_samples_leaf: int  # ✅ Add this
    random_state: int  # ✅ Add this
    target_column: str


In [14]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [15]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.RandomForest  # ✅ Use RandomForest parameters
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        return ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            n_estimators=params.n_estimators,  # ✅ Added
            max_depth=params.max_depth,  # ✅ Added
            min_samples_split=params.min_samples_split,  # ✅ Added
            min_samples_leaf=params.min_samples_leaf,  # ✅ Added
            random_state=params.random_state,  # ✅ Added
            target_column=schema.name
        )


In [18]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        # Load the training and testing datasets
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        # Split into features (X) and target (y)
        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[self.config.target_column]
        test_y = test_data[self.config.target_column]

        # Initialize and train RandomForest model
        rf = RandomForestRegressor(
            n_estimators=self.config.n_estimators,
            max_depth=self.config.max_depth,
            min_samples_split=self.config.min_samples_split,
            min_samples_leaf=self.config.min_samples_leaf,
            random_state=self.config.random_state
        )

        rf.fit(train_x, train_y)

        # Save the trained model
        joblib.dump(rf, os.path.join(self.config.root_dir, self.config.model_name))


In [17]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()  # ✅ Get the ModelTrainerConfig
    model_trainer = ModelTrainer(config=model_trainer_config)  # ✅ Create ModelTrainer instance
    model_trainer.train()  # ✅ Call train method
except Exception as e:
    print(f"Error occurred: {e}")  # Print the error for debugging
    raise e  # Re-raise exception


[2025-04-04 15:21:01,898: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-04-04 15:21:01,909: INFO: common: yaml file: params.yaml loaded successfully]
[2025-04-04 15:21:01,925: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-04-04 15:21:01,931: INFO: common: created directory at: artifacts]
[2025-04-04 15:21:01,936: INFO: common: created directory at: artifacts/model_trainer]
